In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier

In [3]:
import joblib

model_url = joblib.load('/home/ancientsoftware/iot-threat-detector/models/url_model.pkl')
model_iot = joblib.load('/home/ancientsoftware/iot-threat-detector/models/iot_model.pkl')

print(model_url)
print(model_iot)

RandomForestClassifier(random_state=42)
RandomForestClassifier(random_state=42)


In [4]:
url_example = pd.read_csv('/home/ancientsoftware/iot-threat-detector/data/url_example.csv')
iot_example = pd.read_csv('/home/ancientsoftware/iot-threat-detector/data/iot_example.csv')

print(url_example)
print(iot_example)

   url_length  num_dots  num_hyphens   entropy  tld_au  tld_biz  tld_br  \
0          30        30            0  4.056565   False    False   False   

   tld_ca  tld_com  tld_de  tld_edu  tld_gov  tld_in  tld_info  tld_net  \
0   False     True   False    False    False   False     False    False   

   tld_org  tld_other  tld_pl  tld_ru  tld_uk  
0    False      False   False   False   False  
   MI_dir_L5_weight  MI_dir_L5_mean  MI_dir_L5_variance  MI_dir_L3_weight  \
0               1.0            60.0                 0.0               1.0   

   MI_dir_L3_mean  MI_dir_L3_variance  MI_dir_L1_weight  MI_dir_L1_mean  \
0            60.0                 0.0               1.0            60.0   

   MI_dir_L1_variance  MI_dir_L0.1_weight  ...  HpHp_L0.1_radius  \
0                 0.0                 1.0  ...               0.0   

   HpHp_L0.1_covariance  HpHp_L0.1_pcc  HpHp_L0.01_weight  HpHp_L0.01_mean  \
0                   0.0            0.0                1.0             60.0   

  

In [5]:
url_proba = model_url.predict_proba(url_example)
iot_proba = model_iot.predict_proba(iot_example)

print(url_proba)
print(iot_proba)

[[0.25970707 0.74029293]]
[[0. 1.]]


In [6]:
url_risk = url_proba[0][0]  # probability of "bad"
traffic_risk = iot_proba[0][1]  # probability of "attack"

print(url_risk, traffic_risk)

0.2597070661159948 1.0


In [7]:
combined_score = (0.65 * traffic_risk) + (0.35 * url_risk)
print(combined_score)

0.7408974731405982


In [8]:
def risk(combined_score):
    if combined_score < 0.3:
        return "Low risk"
    elif combined_score < 0.7:
        return "Medium risk"
    else:
        return "High risk"

In [9]:
print(risk(combined_score))

High risk


In [17]:
def assess_risk(url_row, traffic_row):
    url_proba = model_url.predict_proba(url_row)
    iot_proba = model_iot.predict_proba(traffic_row)
    url_risk = url_proba[0][0]
    traffic_risk = iot_proba[0][1]
    
    combined_score = (0.65 * traffic_risk) + (0.35 * url_risk)
    
    if combined_score < 0.3:
        label =  "Low risk"
    elif combined_score < 0.7:
        label =  "Medium risk"
    else:
        label =  "High risk"

    return combined_score, label

In [18]:
print(assess_risk(url_example, iot_example))

(np.float64(0.7408974731405982), 'High risk')


In [19]:
score, label = assess_risk(url_example, iot_example)
print(f"Combined score: {score:.2f} — {label}")

Combined score: 0.74 — High risk


In [20]:
url_example_2 = pd.read_csv('/home/ancientsoftware/iot-threat-detector/data/url_example_2.csv')
iot_example_2 = pd.read_csv('/home/ancientsoftware/iot-threat-detector/data/iot_example_2.csv')

score_2, label_2 = assess_risk(url_example_2, iot_example_2)
print(f"Combined score: {score_2:.2f} — {label_2}")

Combined score: 0.09 — Low risk


Tested the combined pipeline on two contrasting cases. A known-benign URL paired with known-benign traffic produced a low risk score (0.09), while a borderline URL paired with confirmed attack traffic produced a high risk score (0.74), correctly flagging the case as high-risk due to the heavier weighting placed on active traffic behavior. This demonstrates the pipeline reasonably distinguishes between low and high risk situations, though further testing with mismatched cases (e.g., suspicious URL with benign traffic) would help validate the weighting scheme further.